# Day 9 · Exercise 4: Robust Extractor with Retry

**What you'll build:** `extract_with_retry(text: str, schema_class: type, model: str, max_tries: int = 3) -> dict` — a function that wraps schema-guided extraction in a repair loop: on `ValidationError` it appends the error as a user turn and asks the model to correct its own output, up to `max_tries` attempts.

**Why it matters:** A single extraction call will occasionally misformat a field; wrapping it in a validate-and-repair loop turns one-off failures into retries, so your pipeline produces clean typed data instead of crashing on the first imperfect response.

## Your Implementation

In [ ]:
import json
import logging
import ollama
from pydantic import BaseModel, Field, ValidationError

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(message)s",
)
logger = logging.getLogger(__name__)


def extract_with_retry(
    text: str,
    model_class: type,
    model: str,
    max_retries: int = 3,
) -> dict:
    """Extract structured data from text, retrying on ValidationError.

    Sends the source text to the model with a system prompt derived from
    model_class.model_json_schema().  If the model's response fails Pydantic
    validation, the failed JSON and the ValidationError are appended to the
    conversation as an assistant turn and a user turn respectively, and the
    model is asked to return corrected JSON.  This loop repeats up to
    max_retries times.  On success the result is returned as a plain dict.

    Args:
        text:         The source text to extract information from.
        model_class:  A Pydantic BaseModel subclass defining the target schema.
        model:        Ollama model name to use (e.g. "llama3.2").
        max_retries:  Maximum total attempts, including the first (default 3).

    Returns:
        A dict produced by model_class.model_dump() on the validated instance.

    Raises:
        ValidationError: If every attempt fails Pydantic validation.

    Example:
        class Event(BaseModel):
            name: str
            year: int

        extract_with_retry(
            "The 2024 Paris Olympics opened on 26 July.",
            Event,
            model="llama3.2",
        )
        # -> {'name': 'Paris Olympics', 'year': 2024}
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

# ── Shared test schema ────────────────────────────────────────────────────────
from pydantic import BaseModel, Field, ValidationError


class EventRecord(BaseModel):
    name: str = Field(description="Full name of the event")
    date_iso: str = Field(description="Event date in ISO 8601 format, e.g. 2024-07-26")
    venue: str = Field(description="Name of the venue or city")
    capacity: int = Field(description="Venue capacity as a plain integer, no commas or units")
    ticket_price_gbp: int = Field(
        description="Ticket price in GBP as a plain integer (no £ symbol, no decimals)"
    )


TEST_MODEL = "llama3.2"

CLEAN_TEXT = (
    "The London Tech Summit will be held at ExCeL London on 2025-03-15. "
    "The venue holds 5000 attendees and tickets cost 120 GBP."
)


def _run_checks():
    score, total = 0, 4

    # Check 1: function is defined and callable
    try:
        assert callable(extract_with_retry), "extract_with_retry is not defined or not callable"
        print(f"{_PASS} Check 1/{total}: extract_with_retry is defined and callable")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 1/{total}: {e}")
        return

    # Check 2: returns a dict for clean input
    try:
        result = extract_with_retry(CLEAN_TEXT, EventRecord, TEST_MODEL)
        assert isinstance(result, dict), f"expected dict, got {type(result).__name__}"
        print(f"{_PASS} Check 2/{total}: extract_with_retry returns a dict for clean input")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 2/{total}: {e}")
        return

    # Check 3: all required keys present and capacity / ticket_price_gbp are ints
    try:
        required = {"name", "date_iso", "venue", "capacity", "ticket_price_gbp"}
        missing = required - result.keys()
        assert not missing, f"missing keys: {missing}"
        assert isinstance(result["capacity"], int), (
            f"capacity must be int, got {type(result['capacity']).__name__}"
        )
        assert isinstance(result["ticket_price_gbp"], int), (
            f"ticket_price_gbp must be int, got {type(result['ticket_price_gbp']).__name__}"
        )
        print(
            f"{_PASS} Check 3/{total}: dict has all required keys; "
            "capacity and ticket_price_gbp are ints"
        )
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 3/{total}: {e}")

    # Check 4: raises ValidationError when required field is permanently missing
    try:
        # Use a private address with no publicly known capacity
        NO_CAPACITY_TEXT = (
            "The Quarterly Review takes place at 14 Millbrook Lane on 2025-06-01. "
            "Tickets cost 50 GBP."  # capacity is genuinely uninferable
        )

        raised = False
        try:
            extract_with_retry(NO_CAPACITY_TEXT, EventRecord, TEST_MODEL, max_retries=2)
        except ValidationError:
            raised = True
        except Exception:
            pass  # only ValidationError counts — other exceptions do not satisfy this check

        assert raised, (
            "expected ValidationError to propagate when required field is unrecoverable"
        )
        print(
            f"{_PASS} Check 4/{total}: ValidationError raised after exhausting retries "
            "on unrecoverable input"
        )
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 4/{total}: {e}")

    print()
    if score == total:
        print("🎉 Exercise complete!")
    print(f"  {score}/{total} passed." + (" Keep going!" if score < total else ""))


_run_checks()

## Bonus Challenge

Right now the repair message uses `str(e)` to format the `ValidationError`. On a later day you will learn to build structured tool calls so the model can receive errors as JSON rather than prose.

As a preview: replace `str(e)` with `json.dumps(e.errors(), indent=2)` in your repair turn and compare the two formats by adding a `print()` before the second `ollama.chat` call. Notice that the structured version tells the model exactly which field failed, what the input value was, and what type was expected — no ambiguity. Observe whether the model corrects itself faster with the structured error, then remove the `print()` before your final submission.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import json
import logging
import ollama
from pydantic import BaseModel, ValidationError

logger = logging.getLogger(__name__)


def extract_with_retry(
    text: str,
    model_class: type,
    model: str,
    max_retries: int = 3,
) -> dict:
    """Extract structured data from text, retrying on ValidationError."""
    schema = model_class.model_json_schema()
    system_prompt = (
        "You are a data extraction assistant.\n"
        "Extract information from the provided text and return a JSON object "
        "that matches this schema exactly:\n\n"
        f"{json.dumps(schema, indent=2)}\n\n"
        "Return ONLY the JSON object. No prose, no markdown, no explanation."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text},
    ]

    last_error: ValidationError | None = None

    for attempt in range(1, max_retries + 1):
        logger.debug("Extraction attempt %d/%d", attempt, max_retries)

        response = ollama.chat(model=model, messages=messages, format="json")
        raw = response["message"]["content"]
        logger.debug("Raw response: %s", raw[:120])

        try:
            instance = model_class.model_validate_json(raw)
            logger.info("Extraction succeeded on attempt %d", attempt)
            return instance.model_dump()

        except ValidationError as e:
            last_error = e
            logger.warning(
                "Attempt %d/%d failed validation: %s", attempt, max_retries, e.errors()
            )

            if attempt < max_retries:
                messages.append({"role": "assistant", "content": raw})
                messages.append({
                    "role": "user",
                    "content": (
                        f"Your JSON failed validation with these errors:\n\n"
                        f"{e}\n\n"
                        "Return corrected JSON only, matching the schema above. "
                        "Use the original text to find the correct values."
                    ),
                })

    raise last_error
```

**Why this works:** The conversation history carries the model's own failed output (as an `assistant` turn) alongside the exact `ValidationError` text (as a `user` turn), giving the model a precise diagnosis of what went wrong and the original passage to re-read for the correct values. Keeping `format="json"` on every `ollama.chat` call inside the loop guarantees syntactic validity at minimum, so Pydantic only ever has to check field types and constraints — never malformed JSON. When `max_retries` is exhausted the last `ValidationError` is re-raised, letting the caller decide whether to skip, log, or escalate the record rather than swallowing the failure silently.
</details>